<a href="https://colab.research.google.com/github/Saiyyam24/deeplearning_practice_project/blob/main/age_gender_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jangedoo/utkface-new")

print("Path to dataset files:", path)

100%|██████████| 331M/331M [00:01<00:00, 202MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/jangedoo/utkface-new/versions/1


In [ ]:

folder_path = '/root/.cache/kagglehub/datasets/jangedoo/utkface-new/versions/1/UTKFace'


In [ ]:
import os

In [ ]:


age = []
gender = []
file_path = []

for file in os.listdir(folder_path):
  file_path.append(os.path.join(file))
  age.append(file.split('_')[0])
  gender.append(file.split('_')[1])



In [ ]:
len(age)

23708

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame({"age":age,"gender":gender,"file_path":file_path})
df['age'] = df['age'].astype(int)
df['gender'] = df['gender'].astype(int)

In [ ]:
df

,age,gender,file_path
0,30,0,30_0_0_20170117190708466.jpg.chip.jpg
1,28,1,28_1_0_20170116221705370.jpg.chip.jpg
2,26,1,26_1_4_20170117174512949.jpg.chip.jpg
3,40,1,40_1_2_20170116191629950.jpg.chip.jpg
4,16,1,16_1_0_20170109214138699.jpg.chip.jpg
...,...,...,...
23703,18,1,18_1_4_20170109212430115.jpg.chip.jpg
23704,28,0,28_0_0_20170104202019890.jpg.chip.jpg
23705,26,1,26_1_2_20170116184244368.jpg.chip.jpg
23706,46,1,46_1_3_20170117190202819.jpg.chip.jpg


In [ ]:
df_train = df.sample(frac=0.8, random_state=0).iloc[:20000]
df_test = df.iloc[20000:]

In [ ]:
df_train.shape

(18966, 3)

In [ ]:
df_test.shape

(3708, 3)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
train_generator = ImageDataGenerator(
    rescale=1./25,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)
test_generator = ImageDataGenerator(
    rescale=1./25
)

In [ ]:
train_gen = train_generator.flow_from_dataframe(
    dataframe=df_train,
    directory=folder_path,
    x_col="file_path",
    y_col=["age","gender"],
    target_size=(200,200),
    batch_size=32,
    class_mode="multi_output"
)

Found 18966 validated image filenames.


In [ ]:
test_gen = test_generator.flow_from_dataframe(
    dataframe=df_test,
    directory=folder_path,
    x_col="file_path",
    y_col=["age","gender"],
    target_size=(200,200),
    class_mode="multi_output"
)

Found 3708 validated image filenames.


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import *
from tensorflow.keras.applications.resnet50 import ResNet50


In [ ]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
resnet.trainable = False

In [ ]:
output = resnet.layers[-1].output

In [ ]:
flatten = Flatten()(output)

In [ ]:
dense1 = Dense(512,activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

In [ ]:
dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

In [ ]:
output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid', name='gender')(dense4)

In [ ]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [ ]:
x, y = train_gen[0]

print(type(x), type(y))
print(x.dtype if hasattr(x, "dtype") else "no dtype")
print(y.dtype if hasattr(y, "dtype") else "no dtype")


<class 'numpy.ndarray'> <class 'list'>
float32
no dtype


In [ ]:

model.compile(optimizer='adam',loss={'age':'mse','gender':'binary_crossentropy'},metrics={'age':'mae','gender':'accuracy'})

In [ ]:
def custom_generator(generator):
    while True:
        x_batch, y_batch = next(generator)
        # Ensure y_batch elements are float32 and returned as a tuple
        yield x_batch, (y_batch[0].astype('float32'), y_batch[1].astype('float32'))

train_gen_fixed = custom_generator(train_gen)
test_gen_fixed = custom_generator(test_gen)

In [ ]:
model.fit(
    train_gen_fixed,
    epochs=10,
    steps_per_epoch=len(train_gen),
    validation_data=test_gen_fixed,
    validation_steps=len(test_gen)
)


Epoch 1/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 223s 376ms/step - age_loss: 196.7966 - age_mae: 10.4727 - gender_accuracy: 0.7557 - gender_loss: 0.4943 - loss: 197.2845 - val_age_loss: 167.3219 - val_age_mae: 9.2122 - val_gender_accuracy: 0.8161 - val_gender_loss: 0.3937 - val_loss: 167.8115
Epoch 2/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 209s 352ms/step - age_loss: 177.6342 - age_mae: 9.8620 - gender_accuracy: 0.7789 - gender_loss: 0.4681 - loss: 178.1200 - val_age_loss: 144.7492 - val_age_mae: 8.9944 - val_gender_accuracy: 0.8211 - val_gender_loss: 0.3840 - val_loss: 145.1332
Epoch 3/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 212s 357ms/step - age_loss: 174.1677 - age_mae: 9.7420 - gender_accuracy: 0.7784 - gender_loss: 0.4650 - loss: 174.5992 - val_age_loss: 161.1660 - val_age_mae: 9.7707 - val_gender_accuracy: 0.8225 - val_gender_loss: 0.3922 - val_loss: 161.5582
Epoch 4/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 209s 352ms/step - age_loss: 173.2446 - age_mae: 9.7149 - gender_accuracy: 0.7817 - gender_loss: 0.4612 - lo